# Сравнение двух реализаций SAE

Сравниваем:
1. Вашу реализацию (`SparseAutoencoder`) - вычитает `b_dec` перед кодированием
2. SAE-lens реализацию (`StandardSAE`) - использует `process_sae_in`


In [1]:
import torch
import numpy as np
from sae_training.sparse_autoencoder import SparseAutoencoder
from sae_lens.saes.standard_sae import StandardSAE, StandardSAEConfig
from sae_training.config import LanguageModelSAERunnerConfig
import torch.nn as nn


In [2]:
import os
from types import SimpleNamespace
from typing import Dict, List, Optional, Sequence
from typing import Any, Dict, Tuple
import torch
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download, HfApi
from safetensors.torch import load_file, save_file
import os

from sae_lens import HookedSAETransformer, StandardSAE, SAEConfig
from safetensors.torch import load_file as safetensors_load_file


import os
from types import SimpleNamespace
from typing import Dict, List, Optional, Sequence

import torch
from safetensors.torch import load_file as safetensors_load_file

from sae_lens import SAE, SAEConfig


def _dtype_to_cfg_str(dtype: torch.dtype | str | None) -> str:
    if dtype is None:
        return "torch.float32"
    if isinstance(dtype, torch.dtype):
        return str(dtype)
    return dtype


def _device_to_cfg_str(device: torch.device | str | None) -> str:
    if device is None:
        return "cpu"
    if isinstance(device, torch.device):
        return str(device)
    return device


def _build_sae_cfg_from_training(
    *,
    model_name: str,
    hook_layer: int,
    d_in: int,
    d_sae: int,
    context_size: int = 128,
    dataset_path: str = "ashaba1in/small_openwebtext",
    hook_name: str = "blocks.{layer}.hook_{target}",  # hook_mlp_in  hook_resid_pre,
    target: str = "resid_pre",  # mid_pre, resid_pre,
    device: torch.device | str | None = "cuda",
    dtype: torch.dtype | str | None = "torch.float32",
    hook_head_index: Optional[int] = None,
) -> SAEConfig:
    target = "mlp_in" if target == "mid_pre" else "resid_pre"
    
    cfg_dict = {
        "architecture": "standard",
        "d_in": int(d_in),
        "d_sae": int(d_sae),
        "activation_fn_str": "relu",
        "activation_fn_kwargs": {},
        "apply_b_dec_to_input": True,
        "finetuning_scaling_factor": False,
        "context_size": int(context_size),
        "model_name": model_name,
        "hook_name": hook_name.format(layer=hook_layer, target=target),
        "hook_layer": int(hook_layer),
        "hook_head_index": hook_head_index,
        "prepend_bos": False,
        "dataset_path": dataset_path,
        "dataset_trust_remote_code": False,
        "normalize_activations": "none",
        "dtype": _dtype_to_cfg_str(dtype),
        "device": _device_to_cfg_str(device),
        "sae_lens_training_version": None,
        "neuronpedia_id": None,
        "model_from_pretrained_kwargs": {},
        "seqpos_slice": (None,),
    }
    return SAEConfig.from_dict(cfg_dict)

In [3]:
layer = 10
hf_model = "ExplosionNuclear/Llama-2.3-3B-Instruct-special-merged-with-19-exp"
repo_id = "Lucid-Layers-Inc/Llama-2.3-3B-Instruct-special-merged-with-19-exp-hook_resid_pre-SAE"
hook_target = "resid_pre"
#repo_id = repo_id.replace("special", "special-merged") if "special-merged" in hf_model else repo_id
# local_path = hf_hub_download(
#     repo_id=repo_id,
#     filename=f"ExplosionNuclear-Llama-2.3-3B-Instruct-special-merged-with-19-exp_layer-10.hook_resid_pre_30720.safetensors",
#     repo_type="model",
# )

local_path = "/workspace-SR008.nfs2/nachevsky/SAE-trainer/checkpoints/0o9mz2ca/final_sae_group_ExplosionNuclear/Llama-2.3-3B-Instruct-special-merged-with-19-exp_blocks.10.hook_resid_pre_30720/ExplosionNuclear-Llama-2.3-3B-Instruct-special-merged-with-19-exp_layer-10.hook_resid_pre_30720.safetensors"
weights = load_file(local_path)
hidden_dim, d_in = weights["W_dec"].shape

In [4]:
from dotenv import load_dotenv
from typing import Any
from omegaconf import OmegaConf
import fire

from sae_training.config import LanguageModelSAERunnerConfig
from sae_training.lm_runner import language_model_sae_runner
from sae_training.utils import _parse_dtype, _parse_device, _parse_hook_point_layer, get_hub_repo_id, get_project_name


def build_config(config: str = "configs/train.yaml", **overrides: Any) -> LanguageModelSAERunnerConfig:
    base_cfg = OmegaConf.load(config)
    override_cfg = OmegaConf.create(overrides) if overrides else OmegaConf.create({})
    override_cfg.hook_point_layer = _parse_hook_point_layer(override_cfg.hook_point_layer)
    merged: Any = OmegaConf.merge(base_cfg, override_cfg)
    merged.hub_repo_id = get_hub_repo_id(merged.model_name, merged.hook_point)
    merged.wandb_project = get_project_name(merged.model_name, merged.hook_point)

    print(merged.hook_point_layer)
    # Map dtype and device
    dtype_str = merged.get("dtype", "float32")
    device_str = merged.get("device", "auto")

    data: dict[str, Any] = OmegaConf.to_container(merged, resolve=True)  # type: ignore[assignment]
    data["dtype"] = _parse_dtype(dtype_str)
    data["device"] = _parse_device(device_str)

    return LanguageModelSAERunnerConfig(**data)

In [26]:

# Конфиг для вашей реализации
your_config = build_config(config="configs/try.yaml", hook_point_layer=[10], hook_point="blocks.{layer}.hook_resid_pre")

# Конфиг для SAE-lens реализации
saelens_config = _build_sae_cfg_from_training(d_sae=hidden_dim, d_in=d_in, hook_layer=layer, model_name=hf_model, target=hook_target)


[10]
30720-layers-10-10-L1-1e-06-LR-0.0003-Tokens-8.000e+06
n_tokens_per_buffer (millions): 0.131072
Lower bound: n_contexts_per_buffer (millions): 0.001024
Total training steps: 988
Total wandb updates: 98
n_tokens_per_feature_sampling_window (millions): 2072.576
n_tokens_per_dead_feature_window (millions): 1036.288
We will reset the sparsity calculation 0 times.
Number tokens in sparsity calculation window: 1.62e+07


In [6]:
# one_sae = SparseAutoencoder(your_config)

In [ ]:
# Создаем обе модели и копируем веса
your_sae = SparseAutoencoder(your_config)
saelens_sae = StandardSAE(saelens_config)



/home/jovyan/micromamba/envs/ilya/lib/python3.11/site-packages/sae_lens/saes/sae.py:249: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [27]:
print("Models created")
print(f"Your SAE parameters: {sum(p.numel() for p in your_sae.parameters())}")
print(f"SAE-lens parameters: {sum(p.numel() for p in saelens_sae.parameters())}")

with torch.no_grad():

    your_sae.W_enc.data = weights["W_enc"].data.clone()
    your_sae.b_enc.data = weights["b_enc"].data.clone()
    your_sae.W_dec.data = weights["W_dec"].data.clone()
    your_sae.b_dec.data = weights["b_dec"].data.clone()

    # your_sae.W_enc.data = one_sae.W_enc.data.clone()
    # your_sae.b_enc.data = one_sae.b_enc.data.clone()
    # your_sae.W_dec.data = one_sae.W_dec.data.clone()
    # your_sae.b_dec.data = one_sae.b_dec.data.clone()

    saelens_sae.W_enc.data = your_sae.W_enc.data.clone()
    saelens_sae.b_enc.data = your_sae.b_enc.data.clone()
    saelens_sae.W_dec.data = your_sae.W_dec.data.clone()
    saelens_sae.b_dec.data = your_sae.b_dec.data.clone()

print("\\nWeights copied from your SAE to SAE-lens SAE")
print("Weight differences:")
print(f"W_enc max diff: {torch.max(torch.abs(your_sae.W_enc - saelens_sae.W_enc)).item():.2e}")
print(f"b_enc max diff: {torch.max(torch.abs(your_sae.b_enc - saelens_sae.b_enc)).item():.2e}")
print(f"W_dec max diff: {torch.max(torch.abs(your_sae.W_dec - saelens_sae.W_dec)).item():.2e}")
print(f"b_dec max diff: {torch.max(torch.abs(your_sae.b_dec - saelens_sae.b_dec)).item():.2e}")


Models created
Your SAE parameters: 188777472
SAE-lens parameters: 188777472
\nWeights copied from your SAE to SAE-lens SAE
Weight differences:
W_enc max diff: 0.00e+00
b_enc max diff: 0.00e+00
W_dec max diff: 0.00e+00
b_dec max diff: 0.00e+00


## Eval Reproduction 

In [4]:
from train_sae import build_config

your_config = build_config(config="configs/try.yaml", hook_point_layer=[10], hook_point="blocks.{layer}.hook_resid_pre")


[10]
30720-layers-10-10-L1-1e-06-LR-0.0003-Tokens-8.000e+06
n_tokens_per_buffer (millions): 0.131072
Lower bound: n_contexts_per_buffer (millions): 0.001024
Total training steps: 988
Total wandb updates: 98
n_tokens_per_feature_sampling_window (millions): 2072.576
n_tokens_per_dead_feature_window (millions): 1036.288
We will reset the sparsity calculation 0 times.
Number tokens in sparsity calculation window: 1.62e+07


In [5]:
from sae_training.utils import LMSparseAutoencoderSessionloader

loader = LMSparseAutoencoderSessionloader(your_config)
model = loader.get_model(your_config.model_name)
model.to(your_config.device)

activation_store = loader.get_activations_loader(your_config, model)

model_name ExplosionNuclear/Llama-2.3-3B-Instruct-special-merged-with-19-exp


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Included rotary_base = 500000.0
Loading weights:
  - Missing keys: 112
  - Unexpected keys: 0
First 5 missing keys:
    1. blocks.0.attn.mask
    2. blocks.0.attn.IGNORE
    3. blocks.0.attn.rotary_sin
    4. blocks.0.attn.rotary_cos
    5. blocks.1.attn.mask
✓ Model created and weights loaded!
Moving model to device:  cuda


In [6]:
your_sae = SparseAutoencoder(your_config)


with torch.no_grad():

    your_sae.W_enc.data = weights["W_enc"].data.clone()
    your_sae.b_enc.data = weights["b_enc"].data.clone()
    your_sae.W_dec.data = weights["W_dec"].data.clone()
    your_sae.b_dec.data = weights["b_dec"].data.clone()

In [7]:
batch_size = your_config.train_batch_size
n_checkpoints = your_config.n_checkpoints

total_training_tokens = your_sae.cfg.total_training_tokens
total_training_steps = total_training_tokens // batch_size
n_training_steps = 0
n_training_tokens = 0

checkpoint_thresholds = []
if n_checkpoints > 0:
    checkpoint_thresholds = list(
        range(0, total_training_tokens,
              total_training_tokens // n_checkpoints)
    )[1:]

all_layers = your_sae.cfg.hook_point_layer

In [8]:
your_sae.cfg.hook_point_layer = 10

In [13]:
# from sae_training.train_sae_on_language_model import _build_train_context, _init_sae_group_b_decs

# train_contexts = [
#     _build_train_context(your_sae, total_training_steps)
# ]
# _init_sae_group_b_decs([your_sae], activation_store, all_layers)

In [9]:
sparse_autoencoder = your_sae
sparse_autoencoder.eval()

hook_point = sparse_autoencoder.cfg.hook_point
hook_point_layer = sparse_autoencoder.cfg.hook_point_layer
hook_point_head_index = sparse_autoencoder.cfg.hook_point_head_index

### Evals
eval_tokens = activation_store.get_batch_tokens()


In [10]:
sparse_autoencoder.cfg.hook_point = sparse_autoencoder.cfg.hook_point.format(layer=10)

In [11]:
sparse_autoencoder.cfg.hook_point

'blocks.10.hook_resid_pre'

In [12]:
sparse_autoencoder.cfg.logger_backend = ""

In [13]:
sparse_autoencoder = sparse_autoencoder.to(your_config.device)

In [14]:
_ = model.eval()

In [15]:
with torch.no_grad():
    hook_point = sparse_autoencoder.cfg.hook_point
    hook_point_layer = sparse_autoencoder.cfg.hook_point_layer
    hook_point_head_index = sparse_autoencoder.cfg.hook_point_head_index
    hook_point, hook_point_layer, hook_point_head_index
    
    eval_tokens = activation_store.get_batch_tokens()
    
    _, cache = model.run_with_cache(
        eval_tokens,
        prepend_bos=False,
        names_filter=[hook_point],
    )
    
    original_act = cache[sparse_autoencoder.cfg.hook_point]
    sae_out, _feature_acts, *_ = sparse_autoencoder(original_act)

In [34]:
del cache
torch.cuda.empty_cache()

In [16]:
torch.norm(original_act - sae_out,dim=-1).mean()

tensor(1.7598, device='cuda:0')

In [36]:
from sae_training.evals import run_evals

outs = run_evals(
    sparse_autoencoder,
    activation_store,
    model,
    n_training_steps,
)
print(outs)

{'metrics/l2_norm': 127.7140884399414, 'metrics/l2_ratio': 0.908531129360199, 'metrics/l2_normalized_error': 69.60083770751953}


## Manual testing

In [15]:
saelens_sae.cuda(), your_sae.cuda()

(StandardSAE(
   (activation_fn): ReLU()
   (hook_sae_input): HookPoint()
   (hook_sae_acts_pre): HookPoint()
   (hook_sae_acts_post): HookPoint()
   (hook_sae_output): HookPoint()
   (hook_sae_recons): HookPoint()
   (hook_sae_error): HookPoint()
 ),
 SparseAutoencoder(
   (hook_sae_in): HookPoint()
   (hook_hidden_pre): HookPoint()
   (hook_hidden_post): HookPoint()
   (hook_sae_out): HookPoint()
 ))

In [16]:
# Создаем тестовые данные и прогоняем через обе модели
torch.manual_seed(42)
x = torch.randn(10, 3072, dtype=torch.float32).cuda()

print(f"Test input shape: {x.shape}")
print(f"Test input mean: {x.mean().item():.4f}, std: {x.std().item():.4f}")

# Прогоняем через обе модели
with torch.no_grad():
    # Ваша реализация
    your_output = your_sae(x)
    
    # SAE-lens реализация  
    saelens_features = saelens_sae.encode(x)
    saelens_reconstruction = saelens_sae.decode(saelens_features)

print(f"\\nYour SAE output shape: {your_output.sae_out.shape}")
print(f"SAE-lens reconstruction shape: {saelens_reconstruction.shape}")
print(f"Your SAE features shape: {your_output.feature_acts.shape}")
print(f"SAE-lens features shape: {saelens_features.shape}")


Test input shape: torch.Size([10, 3072])
Test input mean: 0.0047, std: 1.0040
\nYour SAE output shape: torch.Size([10, 3072])
SAE-lens reconstruction shape: torch.Size([10, 3072])
Your SAE features shape: torch.Size([10, 30720])
SAE-lens features shape: torch.Size([10, 30720])


In [17]:
# Пошаговое сравнение реализаций
print("=== ПОШАГОВОЕ СРАВНЕНИЕ ===")

# 1. Входные данные после preprocessing
your_sae_in = x - your_sae.b_dec  # Ваша реализация
saelens_sae_in = saelens_sae.process_sae_in(x)  # SAE-lens

print(f"\\n1. SAE input preprocessing:")
print(f"   Your (x - b_dec) mean: {your_sae_in.mean().item():.6f}")
print(f"   SAE-lens process_sae_in mean: {saelens_sae_in.mean().item():.6f}")
print(f"   Max difference: {torch.max(torch.abs(your_sae_in - saelens_sae_in)).item():.2e}")

# 2. Кодирование (pre-activation)
your_hidden_pre = your_sae_in @ your_sae.W_enc + your_sae.b_enc
saelens_hidden_pre = saelens_sae_in @ saelens_sae.W_enc + saelens_sae.b_enc

print(f"\\n2. Encoding (pre-activation):")
print(f"   Your hidden_pre mean: {your_hidden_pre.mean().item():.6f}")
print(f"   SAE-lens hidden_pre mean: {saelens_hidden_pre.mean().item():.6f}")
print(f"   Max difference: {torch.max(torch.abs(your_hidden_pre - saelens_hidden_pre)).item():.2e}")

# 3. Активация (ReLU)
your_features = torch.relu(your_hidden_pre)

print(f"\\n3. Features (post-ReLU):")
print(f"   Your features mean: {your_features.mean().item():.6f}")
print(f"   SAE-lens features mean: {saelens_features.mean().item():.6f}")
print(f"   Max difference: {torch.max(torch.abs(your_features - saelens_features)).item():.2e}")

# 4. Финальные реконструкции
print(f"\\n4. Final reconstructions:")
print(f"   Your reconstruction mean: {your_output.sae_out.mean().item():.6f}")
print(f"   SAE-lens reconstruction mean: {saelens_reconstruction.mean().item():.6f}")
print(f"   Max difference: {torch.max(torch.abs(your_output.sae_out - saelens_reconstruction)).item():.2e}")


=== ПОШАГОВОЕ СРАВНЕНИЕ ===
\n1. SAE input preprocessing:
   Your (x - b_dec) mean: 0.048641
   SAE-lens process_sae_in mean: 0.048641
   Max difference: 0.00e+00
\n2. Encoding (pre-activation):
   Your hidden_pre mean: 0.210911
   SAE-lens hidden_pre mean: 0.210911
   Max difference: 0.00e+00
\n3. Features (post-ReLU):
   Your features mean: 0.375336
   SAE-lens features mean: 0.375336
   Max difference: 0.00e+00
\n4. Final reconstructions:
   Your reconstruction mean: 0.002092
   SAE-lens reconstruction mean: 0.002092
   Max difference: 0.00e+00


In [18]:
# Финальное сравнение и вывод
print("=== ФИНАЛЬНОЕ СРАВНЕНИЕ ===")

your_recon_error = torch.mean((x - your_output.sae_out) ** 2)
saelens_recon_error = torch.mean((x - saelens_reconstruction) ** 2)

print(f"\\nReconstruction MSE:")
print(f"   Your SAE: {your_recon_error.item():.6f}")
print(f"   SAE-lens: {saelens_recon_error.item():.6f}")
print(f"   Difference: {abs(your_recon_error.item() - saelens_recon_error.item()):.2e}")

# Sparsity comparison
your_sparsity = (your_output.feature_acts > 0).float().mean()
saelens_sparsity = (saelens_features > 0).float().mean()

print(f"\\nSparsity (fraction of active features):")
print(f"   Your SAE: {your_sparsity.item():.4f}")
print(f"   SAE-lens: {saelens_sparsity.item():.4f}")
print(f"   Difference: {abs(your_sparsity.item() - saelens_sparsity.item()):.6f}")

# Общий вывод
max_diff = torch.max(torch.abs(your_output.sae_out - saelens_reconstruction)).item()
print(f"\\n=== ВЫВОД ===")
if max_diff < 1e-6:
    print(f"✅ Реализации ИДЕНТИЧНЫ (max diff: {max_diff:.2e})")
elif max_diff < 1e-3:
    print(f"⚠️  Реализации очень похожи, небольшие различия (max diff: {max_diff:.2e})")
else:
    print(f"❌ Реализации РАЗЛИЧАЮТСЯ значительно (max diff: {max_diff:.2e})")
    print("   Нужно проверить различия в архитектуре")
    
# Исследуем process_sae_in подробнее
print(f"\\n=== АНАЛИЗ process_sae_in ===")
print(f"b_dec в вашей модели: {your_sae.b_dec[:5]}")  # первые 5 элементов
print(f"b_dec в SAE-lens: {saelens_sae.b_dec[:5]}")
print(f"Равны ли b_dec? {torch.allclose(your_sae.b_dec, saelens_sae.b_dec)}")

# Проверим, что делает process_sae_in
manual_process = x  # возможно, process_sae_in просто возвращает x?
print(f"\\nТест: process_sae_in vs identity:")
print(f"   process_sae_in(x) == x? {torch.allclose(saelens_sae_in, x)}")
print(f"   Max diff from identity: {torch.max(torch.abs(saelens_sae_in - x)).item():.2e}")


=== ФИНАЛЬНОЕ СРАВНЕНИЕ ===
\nReconstruction MSE:
   Your SAE: 0.507038
   SAE-lens: 0.507038
   Difference: 0.00e+00
\nSparsity (fraction of active features):
   Your SAE: 0.6290
   SAE-lens: 0.6290
   Difference: 0.000000
\n=== ВЫВОД ===
✅ Реализации ИДЕНТИЧНЫ (max diff: 0.00e+00)
\n=== АНАЛИЗ process_sae_in ===
b_dec в вашей модели: tensor([-2.3965, -1.0994, -2.2989, -0.3474,  2.2648], device='cuda:0',
       grad_fn=<SliceBackward0>)
b_dec в SAE-lens: tensor([-2.3965, -1.0994, -2.2989, -0.3474,  2.2648], device='cuda:0',
       grad_fn=<SliceBackward0>)
Равны ли b_dec? True
\nТест: process_sae_in vs identity:
   process_sae_in(x) == x? False
   Max diff from identity: 6.32e+00
